# EE-3.2: Transmission line models — capacitance, line charging and the Ferranti effect

## Objectives

At the end of this notebook you should be able to:

- explain the three physical parameters of a line — resistance, inductance and **capacitance** — and their typical values for overhead lines and cables
- build the **nominal π model** and its two-port (ABCD) description
- compute a line's **charging** reactive power, and explain why long AC cables are impractical
- predict the **Ferranti effect** — a voltage *rise* at the open end of a long line — and check it against pandapower
- use the **surge impedance loading** (SIL) to tell whether a loaded line produces or consumes reactive power
- judge when the simple π model is accurate, and validate the **exact long-line model** against a chain of many short sections

Until now every line was a series impedance; the capacitance between conductors and ground was set to zero (`c_nf_per_km=0` in pandapower). For short lines that is fine. For long overhead lines and for cables it is not — and the effects are some of the most surprising in power systems.

In [1]:
import warnings

import numpy as np
import pandas as pd
import pandapower as pp

warnings.filterwarnings("ignore", category=FutureWarning)
omega = 2 * np.pi * 50.0

### What a line is made of

A three-phase line has three distributed physical effects along its whole length:

| Parameter | Physics | Per-phase value, 400 kV overhead line | 110 kV underground cable |
|---|---|---|---|
| resistance r | conductor heating (EE-1) | ≈ 0.03 Ω/km | ≈ 0.06 Ω/km |
| reactance x = ωL | magnetic field around the current (EE-2.1) | ≈ 0.3 Ω/km | ≈ 0.12 Ω/km |
| capacitance c | electric field between conductors and ground | ≈ 12 nF/km | ≈ 200 nF/km |

These are typical orders of magnitude, used throughout this lesson; real lines are specified in the network data (pandapower's standard types, TSO data). Overhead conductors are far apart and far from the ground, so their capacitance is small; a cable's conductor is separated from its grounded sheath by a few centimetres of insulation, so its capacitance is more than ten times larger.

In [2]:
r, x, c = 0.03, 0.30, 12e-9          # 400 kV overhead line: ohm/km, ohm/km, F/km
V_kV = 400.0


def nominal_pi(length_km, r=r, x=x, c=c):
    """Series impedance Z (ohm) and total shunt admittance Y (S) of a line."""
    return (r + 1j * x) * length_km, 1j * omega * c * length_km

### The nominal π model

Lump the parameters of the whole line into three elements: the series impedance **Z = (r + jx)·ℓ** in the middle, and half of the shunt admittance **Y = jωc·ℓ** at each end — a shape like the Greek letter π. This is the model pandapower and PyPSA use for every line.

A line is a **two-port**: two voltages and two currents. Its sending-end quantities follow from the receiving-end ones through the **ABCD matrix**:

**V_s = A·V_r + B·I_r,  I_s = C·V_r + D·I_r**

with A = D = 1 + ZY/2, B = Z, C = Y(1 + ZY/4) for the π model. Chains of elements simply multiply their ABCD matrices — useful below.

In [3]:
def abcd_pi(Z, Y):
    """ABCD matrix of a nominal pi section."""
    A = 1 + Z * Y / 2
    return np.array([[A, Z], [Y * (1 + Z * Y / 4), A]])


Z300, Y300 = nominal_pi(300)
M = abcd_pi(Z300, Y300)
assert np.isclose(np.linalg.det(M), 1.0), "a passive, reciprocal two-port has AD - BC = 1"
print(f"300 km: Z = {Z300:.1f} ohm, Y = {Y300.imag * 1e3:.3f}j mS, A = {M[0, 0]:.5f}")

300 km: Z = 9.0+90.0j ohm, Y = 1.131j mS, A = 0.94911+0.00509j


The determinant check AD − BC = 1 holds for every passive, reciprocal two-port — a quick test that the matrix was built correctly.

### Line charging

Energise a line, and its capacitance draws a charging current even with nothing connected at the far end. That current leads the voltage by 90°: the line **produces** reactive power, roughly Q ≈ V²·B with B = ωc·ℓ (line-to-line kV, three-phase Mvar, EE-3.1):

In [4]:
rows = []
for name, length, c_km in [("400 kV overhead", 100, 12e-9), ("400 kV overhead", 300, 12e-9),
                           ("400 kV overhead", 600, 12e-9), ("110 kV cable", 50, 200e-9)]:
    V = 400.0 if "400" in name else 110.0
    B = omega * c_km * length
    rows.append({"line": name, "length_km": length, "B_mS": B * 1e3, "charging_Mvar": V**2 * B})
pd.DataFrame(rows).round(2)

              line  length_km  B_mS  charging_Mvar
0  400 kV overhead        100  0.38          60.32
1  400 kV overhead        300  1.13         180.96
2  400 kV overhead        600  2.26         361.91
3     110 kV cable         50  3.14          38.01

A 300 km overhead line produces about 180 Mvar on its own — more than many generators. And a 50 km, 110 kV cable produces 38 Mvar — as much as roughly 830 km of 110 kV overhead line would, because its capacitance per kilometre is about 17 times larger. The charging current also flows through the cable's conductor and uses up its thermal capacity, so beyond some tens of kilometres an AC cable carries mostly its own charging current. That is why long subsea and underground connections — offshore wind farms, interconnectors — use **HVDC**: with direct current there is no charging current in steady state.

### The Ferranti effect: higher voltage at the open end

With no load at the far end (I_r = 0), the two-port gives V_s = A·V_r, so **V_r = V_s / A**. For a lossless line A = 1 − ω²LCℓ²/2 < 1 — the receiving voltage is *higher* than the sending voltage. The charging current flows through the line's inductance, and the voltage drop across an inductance carrying a *leading* current is a voltage *rise*. Compute it, and check against pandapower with the capacitance switched on:

In [5]:
rows = []
for length in [100, 300, 600]:
    Z, Y = nominal_pi(length)
    ours = abs(1 / abcd_pi(Z, Y)[0, 0])
    net = pp.create_empty_network()
    b1, b2 = pp.create_bus(net, vn_kv=400), pp.create_bus(net, vn_kv=400)
    pp.create_ext_grid(net, b1, vm_pu=1.0)
    pp.create_line_from_parameters(net, b1, b2, length_km=length, r_ohm_per_km=r, x_ohm_per_km=x,
                                   c_nf_per_km=c * 1e9, max_i_ka=3.0)
    pp.runpp(net, numba=False)
    rows.append({"length_km": length, "Vr_over_Vs_pi_model": ours, "pandapower_vm_pu": net.res_bus.vm_pu[b2],
                 "Vr_kV": 400 * ours, "source_absorbs_Mvar": -net.res_ext_grid.q_mvar[0]})
ferranti = pd.DataFrame(rows)
assert np.allclose(ferranti.Vr_over_Vs_pi_model, ferranti.pandapower_vm_pu, atol=1e-8)
ferranti.round(4)

   length_km  Vr_over_Vs_pi_model  pandapower_vm_pu     Vr_kV  \
0        100               1.0057            1.0057  402.2747   
1        300               1.0536            1.0536  421.4431   
2        600               1.2552            1.2552  502.0805   

   source_absorbs_Mvar  
0              60.4901  
1             185.8047  
2             408.0175  

Matched to eight decimals. An unloaded 300 km line delivers **421 kV** at its far end from a 400 kV source, a rise of 5.4 %; at 600 km the rise is 25 %, far beyond what equipment tolerates. The source must also **absorb** the reactive power the line produces. This is a real operating problem: at night, when lines are lightly loaded, voltages in transmission grids rise, and operators switch in **shunt reactors** (large inductors) at substations or even take lines out of service to keep voltages down.

### Surge impedance loading: when does a line produce or consume Q?

A line both **produces** reactive power (its capacitance, ∝ V²) and **consumes** it (its inductance, ∝ I², EE-2.2). The two balance at one particular load, the **surge impedance loading**:

**Z_c = √(z / y)  (characteristic impedance),  SIL = V² / |Z_c|**

with z = r + jωL and y = jωc per km. Below the SIL, charging wins: the line exports reactive power and the receiving voltage rises. Above it, the inductance wins: the line consumes reactive power and the voltage sags. Check with a lossless 300 km line carrying increasing loads at unity power factor:

In [6]:
Z_c = np.sqrt((r + 1j * x) / (1j * omega * c))
SIL = V_kV**2 / abs(Z_c)
print(f"characteristic impedance |Z_c| = {abs(Z_c):.0f} ohm, SIL = {SIL:.0f} MW")

rows = []
for P in [200, 400, round(SIL), 700, 900]:
    net = pp.create_empty_network()
    b1, b2 = pp.create_bus(net, vn_kv=400), pp.create_bus(net, vn_kv=400)
    pp.create_ext_grid(net, b1, vm_pu=1.0)
    pp.create_line_from_parameters(net, b1, b2, length_km=300, r_ohm_per_km=0.0, x_ohm_per_km=x,
                                   c_nf_per_km=c * 1e9, max_i_ka=5.0)
    pp.create_load(net, b2, p_mw=P, q_mvar=0.0)
    pp.runpp(net, numba=False)
    rows.append({"load_MW": P, "receiving_vm_pu": net.res_bus.vm_pu[b2], "source_Q_Mvar": net.res_ext_grid.q_mvar[0]})
sil_table = pd.DataFrame(rows)
assert sil_table.receiving_vm_pu.is_monotonic_decreasing
assert abs(sil_table.loc[2, "source_Q_Mvar"]) < 0.05 * sil_table.source_Q_Mvar.abs().max(), "near zero at the SIL"
sil_table.round(3)

characteristic impedance |Z_c| = 283 ohm, SIL = 566 MW


   load_MW  receiving_vm_pu  source_Q_Mvar
0      200            1.048       -164.203
1      400            1.028        -96.091
2      566            0.999          4.571
3      700            0.961        128.546
4      900            0.842        491.646

At 200 MW the line pushes 164 Mvar back into the source and its far end sits 4.8 % above nominal; at the SIL of about 566 MW the source exchanges almost no reactive power and the voltage is flat; at 900 MW the line consumes almost 500 Mvar and the far end sags to 0.84 pu. (On a single π section the balance is not perfectly exact — the distributed line of the next section is.) The SIL is a handy rule of thumb for planners: long overhead lines are usually operated around or below their SIL; cables, with their small Z_c, usually have SILs above their thermal limit, so in practice they always produce reactive power.

### How accurate is one π section? The exact long-line model

Parameters are really spread along the line, not lumped. Solving the line's differential equations exactly gives the **distributed** model, with propagation constant γ = √(z·y):

**A = cosh(γℓ),  B = Z_c·sinh(γℓ),  C = sinh(γℓ) / Z_c**

An independent way to reach the same answer: split the line into N short π sections and multiply their ABCD matrices. As N grows, the chain must converge to the exact model — if both are right.

In [7]:
gamma = np.sqrt((r + 1j * x) * (1j * omega * c))


def abcd_exact(length):
    """ABCD matrix of the distributed-parameter line."""
    gl = gamma * length
    return np.array([[np.cosh(gl), Z_c * np.sinh(gl)], [np.sinh(gl) / Z_c, np.cosh(gl)]])


def abcd_chain(length, n_sections):
    """ABCD matrix of n identical pi sections in series."""
    return np.linalg.matrix_power(abcd_pi(*nominal_pi(length / n_sections)), n_sections)


rows = []
for length in [100, 300, 600]:
    exact = abs(1 / abcd_exact(length)[0, 0])
    row = {"length_km": length, "exact_Vr_over_Vs": exact}
    for n in [1, 10, 100]:
        row[f"{n}_sections_error_pct"] = 100 * (abs(1 / abcd_chain(length, n)[0, 0]) - exact) / exact
    rows.append(row)
accuracy = pd.DataFrame(rows)
assert (accuracy["100_sections_error_pct"].abs() < 1e-3).all(), "the chain converges to the exact model"
accuracy.round(5)

   length_km  exact_Vr_over_Vs  1_sections_error_pct  10_sections_error_pct  \
0        100           1.00568               0.00053                0.00001   
1        300           1.05314               0.04483                0.00044   
2        600           1.24471               0.84255                0.00792   

   100_sections_error_pct  
0                 0.00000  
1                 0.00000  
2                 0.00008  

Two independent models agree: the chain of 100 short sections reproduces the exact distributed line to five decimals at every length. Against that reference, a single π section is excellent up to a few hundred kilometres (0.04 % error at 300 km) but overstates the Ferranti rise by 0.8 % at 600 km — and the error grows quickly with length.

Practical consequence: pandapower and PyPSA model each line as **one** nominal π section (that is why pandapower matched the single-section column above). For very long lines, either split them into several sections in the network data or use the exact model's equivalent π (Z' = B, Y' from A and B). Typical transmission lines between neighbouring substations are much shorter than the 600 km case here, and for them one section is accurate.

## Check your understanding

**Charging and Ferranti**

1. Why do transmission voltages tend to rise at night, and what equipment do operators switch in to counter it?
2. A 30 km, 220 kV cable connects an offshore wind farm. Estimate its charging power with c = 200 nF/km. Where would you place shunt reactors?

**SIL and accuracy**

1. A 400 kV line carries 300 MW in the morning and 900 MW at the evening peak. Does it produce or consume reactive power in each case, and what does that mean for the voltage control at its ends?
2. Why does splitting a line into more π sections make the model more accurate, but never change the result for a line with zero capacitance?